In [16]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.8 MB/s eta 0:00:00


In [17]:
# =========================================================
# INSTALL REQUIRED LIBRARIES
# =========================================================

!pip install reportlab
!pip install pandas
!pip install matplotlib

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import json
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image,
    Table,
    TableStyle
)

from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

print("✅ All Libraries Imported Successfully")

✅ All Libraries Imported Successfully


In [21]:
required_columns = []

In [25]:
# =========================================================
# PROJECT :
# JSON TO STRUCTURED PROFESSIONAL PDF REPORT
# =========================================================

# =========================================================
# INSTALL REQUIRED LIBRARIES
# =========================================================

!pip install reportlab
!pip install pandas
!pip install matplotlib

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image,
    Table,
    TableStyle
)

from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

# =========================================================
# STEP 1 : LOAD JSON FILE
# =========================================================

json_file = "US_STATE_recipes.json"

with open(json_file, "r", encoding="utf-8") as file:

    data = json.load(file)

print("✅ JSON File Loaded Successfully")

# =========================================================
# STEP 2 : CREATE STRUCTURED DATAFRAME
# =========================================================

# ---------------------------------------------------------
# CASE 1 : LIST FORMAT
# ---------------------------------------------------------

if isinstance(data, list):

    # Check if first record is dictionary
    if len(data) > 0 and isinstance(data[0], dict):

        df = pd.json_normalize(data)

    else:

        df = pd.DataFrame(data)

# ---------------------------------------------------------
# CASE 2 : DICTIONARY FORMAT
# ---------------------------------------------------------

elif isinstance(data, dict):

    # Convert dict values properly
    df = pd.DataFrame.from_dict(data, orient='index')

    # Reset Index
    df.reset_index(inplace=True)

# ---------------------------------------------------------
# INVALID FORMAT
# ---------------------------------------------------------

else:

    raise Exception("Unsupported JSON Structure")

print("\n✅ Structured DataFrame Created Successfully")

# =========================================================
# STEP 3 : CLEAN COLUMN NAMES
# =========================================================

df.columns = [str(col).replace(".", "_") for col in df.columns]

print("\n================ AVAILABLE COLUMNS =================")

print(df.columns)

# =========================================================
# STEP 4 : HANDLE EMPTY DATAFRAME
# =========================================================

if df.empty:

    raise Exception("DataFrame is Empty")

# =========================================================
# STEP 5 : SELECT VALID COLUMNS
# =========================================================

# Select first available columns dynamically

max_columns = min(6, len(df.columns))

structured_df = df.iloc[:, :max_columns]

print("\n✅ Structured Columns Selected")

print("\n================ SAMPLE DATA =================")

print(structured_df.head())

# =========================================================
# STEP 6 : HANDLE NULL VALUES
# =========================================================

structured_df = structured_df.fillna("N/A")

# =========================================================
# STEP 7 : EXPORT CSV
# =========================================================

csv_output = "structured_recipes_output.csv"

structured_df.to_csv(csv_output, index=False)

print(f"\n✅ CSV Generated : {csv_output}")

# =========================================================
# STEP 8 : CREATE ANALYTICS CHART
# =========================================================

# Select safe chart column

chart_column = structured_df.columns[0]

chart_data = (
    structured_df[chart_column]
    .astype(str)
    .value_counts()
    .head(10)
)

# Create Chart
plt.figure(figsize=(10, 5))

chart_data.plot(kind='bar')

plt.title(f"Top Values of {chart_column}")

plt.xlabel(chart_column)

plt.ylabel("Count")

plt.tight_layout()

chart_path = "recipes_chart.png"

plt.savefig(chart_path)

plt.close()

print("\n✅ Analytics Chart Generated")

# =========================================================
# STEP 9 : CREATE PDF DOCUMENT
# =========================================================

pdf_file = "US_STATE_Recipes_Report.pdf"

doc = SimpleDocTemplate(
    pdf_file,
    pagesize=letter,
    rightMargin=20,
    leftMargin=20,
    topMargin=20,
    bottomMargin=20
)

styles = getSampleStyleSheet()

elements = []

# =========================================================
# STEP 10 : TITLE
# =========================================================

title = Paragraph(
    """
    <font size=24 color='darkblue'>
    <b>US State Recipes Structured PDF Report</b>
    </font>
    """,
    styles['Title']
)

elements.append(title)

elements.append(Spacer(1, 20))

# =========================================================
# STEP 11 : SUMMARY SECTION
# =========================================================

summary = Paragraph(
    f"""
    <font size=12 color='darkgreen'>

    <b>Total Rows :</b> {structured_df.shape[0]} <br/>
    <b>Total Columns :</b> {structured_df.shape[1]} <br/><br/>

    This report converts nested JSON recipe data
    into a clean structured tabular PDF format.

    </font>
    """,
    styles['BodyText']
)

elements.append(summary)

elements.append(Spacer(1, 20))

# =========================================================
# STEP 12 : ADD CHART
# =========================================================

chart_image = Image(chart_path)

chart_image.drawHeight = 250

chart_image.drawWidth = 450

elements.append(chart_image)

elements.append(Spacer(1, 20))

# =========================================================
# STEP 13 : CREATE TABLE DATA
# =========================================================

table_data = []

# Header Row
header_row = []

for col in structured_df.columns:

    header_row.append(
        Paragraph(
            f"<b>{str(col).upper()}</b>",
            styles['BodyText']
        )
    )

table_data.append(header_row)

# =========================================================
# STEP 14 : ADD DATA ROWS
# =========================================================

for row in structured_df.head(15).values:

    formatted_row = []

    for cell in row:

        text = str(cell)

        # Trim huge content
        if len(text) > 120:

            text = text[:120] + "..."

        formatted_text = Paragraph(
            text,
            styles['BodyText']
        )

        formatted_row.append(formatted_text)

    table_data.append(formatted_row)

# =========================================================
# STEP 15 : DYNAMIC COLUMN WIDTHS
# =========================================================

column_count = len(structured_df.columns)

page_width = 520

dynamic_width = page_width / column_count

col_widths = [dynamic_width] * column_count

# =========================================================
# STEP 16 : CREATE TABLE
# =========================================================

table = Table(
    table_data,
    colWidths=col_widths
)

# =========================================================
# STEP 17 : TABLE STYLING
# =========================================================

table.setStyle(TableStyle([

    # Header Background
    ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),

    # Header Text
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    # Body Background
    ('BACKGROUND', (0, 1), (-1, -1), colors.beige),

    # Grid
    ('GRID', (0, 0), (-1, -1), 1, colors.black),

    # Font
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    # Alignment
    ('VALIGN', (0, 0), (-1, -1), 'TOP'),

    # Padding
    ('BOTTOMPADDING', (0, 0), (-1, 0), 12),

]))

elements.append(table)

elements.append(Spacer(1, 20))

# =========================================================
# STEP 18 : PROJECT SUMMARY
# =========================================================

project_summary = Paragraph(
    """
    <font size=12 color='darkred'>

    <b>Project Features</b><br/><br/>

    ✅ Nested JSON Flattening <br/>
    ✅ Dynamic Column Handling <br/>
    ✅ CSV Export <br/>
    ✅ Analytics Visualization <br/>
    ✅ Professional PDF Report <br/>
    ✅ Structured Tabular Format <br/>
    ✅ Long Text Formatting <br/>
    ✅ Automatic Error Handling

    </font>
    """,
    styles['BodyText']
)

elements.append(project_summary)

# =========================================================
# STEP 19 : BUILD PDF
# =========================================================

doc.build(elements)

print("\n===================================================")

print("✅ PDF REPORT GENERATED SUCCESSFULLY")

print("===================================================\n")

print("Generated Files:")

print("1. structured_recipes_output.csv")

print("2. recipes_chart.png")

print("3. US_STATE_Recipes_Report.pdf")

# =========================================================
# END OF PROJECT
# =========================================================

✅ JSON File Loaded Successfully

✅ Structured DataFrame Created Successfully

================ AVAILABLE COLUMNS =================
Index(['index', 'Contient', 'Country_State', 'cuisine', 'title', 'URL',
       'rating', 'total_time', 'prep_time', 'cook_time', 'description',
       'ingredients', 'instructions', 'nutrients', 'serves'],
      dtype='object')

✅ Structured Columns Selected

================ SAMPLE DATA =================
  index Contient  Country_State   cuisine                           title  \
0     0   Africa            NaN  Missouri         Ground Beef and Cabbage   
1     1   Africa            NaN  Missouri     Old Fashioned Peach Cobbler   
2     2   Africa            NaN  Missouri       St. Louis Toasted Ravioli   
3     3   Africa            NaN  Missouri  Amish Friendship Bread Starter   
4     4   Africa            NaN  Missouri    Simple Fried Morel Mushrooms   

                                                 URL  
0  https://www.allrecipes.com/recipe/229324/